# Employee Promotion Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether an employee will be **promoted** based on HR metrics including training scores, KPIs, tenure, education, and department. The dataset has ~8,000 employees with ~8.5% promotion rate — an imbalanced classification problem relevant to HR analytics.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given employee attributes (department, education, gender, training score, KPIs met, tenure, awards, previous year rating), predict whether they will be promoted (is_promoted = 1).

## 5 · Why This Project Matters

- HR analytics is a growing field with **direct business impact**.
- Promotion prediction helps identify high-potential employees.
- ~8.5% promotion rate creates a realistic imbalanced classification challenge.
- Understanding promotion drivers reveals organizational patterns.
- Fairness and bias auditing are critical for HR ML applications.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~8,000 |
| Features | 11 (department, education, gender, no_of_trainings, age, previous_year_rating, length_of_service, KPIs_met, awards_won, avg_training_score, recruitment_channel) |
| Target | `is_promoted` (binary: 0=not promoted, 1=promoted) |
| Class balance | ~91.5% not promoted, ~8.5% promoted |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic HR analytics dataset for ML practice.
**License**: Educational / public.
**Note**: Inspired by Analytics Vidhya HR Analytics competition.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "is_promoted"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Employee Promotion Prediction


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 8000

departments = ['Sales & Marketing', 'Operations', 'Technology', 'Analytics', 'R&D',
               'Procurement', 'Finance', 'HR', 'Legal']
department = np.random.choice(departments, n, p=[0.2, 0.15, 0.15, 0.1, 0.1, 0.1, 0.08, 0.07, 0.05])
education = np.random.choice(["Below Secondary", "Bachelor's", "Master's & above"], n, p=[0.15, 0.55, 0.3])
gender = np.random.choice(['m', 'f'], n, p=[0.6, 0.4])
no_of_trainings = np.random.poisson(1.2, n).clip(1, 10)
age = np.random.normal(34, 7, n).clip(22, 60).astype(int)
prev_rating = np.random.choice([1, 2, 3, 4, 5], n, p=[0.02, 0.08, 0.3, 0.4, 0.2])
length_of_service = np.random.exponential(5, n).clip(1, 30).astype(int)
kpis_met = np.random.binomial(1, 0.35, n)
awards_won = np.random.binomial(1, 0.03, n)
avg_training_score = np.random.normal(60, 15, n).clip(30, 99).round(0).astype(int)
recruitment = np.random.choice(['sourcing', 'other', 'referred'], n, p=[0.5, 0.3, 0.2])

score = (
    0.8 * kpis_met
    + 2.0 * awards_won
    + 0.02 * (avg_training_score - 50)
    + 0.15 * (prev_rating - 3)
    + 0.05 * (education == "Master's & above").astype(float)
    - 0.02 * no_of_trainings
    + np.random.normal(0, 0.8, n)
)
is_promoted = (score > np.percentile(score, 91.5)).astype(int)

df = pd.DataFrame({
    'department': department, 'education': education, 'gender': gender,
    'no_of_trainings': no_of_trainings, 'age': age, 'previous_year_rating': prev_rating,
    'length_of_service': length_of_service, 'KPIs_met': kpis_met,
    'awards_won': awards_won, 'avg_training_score': avg_training_score,
    'recruitment_channel': recruitment, 'is_promoted': is_promoted,
})
print(f"Dataset shape: {df.shape}")
print(f"Promotion rate: {df['is_promoted'].mean():.2%}")
df.head()

Dataset shape: (8000, 12)
Promotion rate: 8.50%


,department,education,gender,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met,awards_won,avg_training_score,recruitment_channel,is_promoted
0,Technology,Master's & above,m,2,36,3,5,0,0,77,other,0
1,Legal,Bachelor's,m,3,26,3,2,0,0,71,other,0
2,Procurement,Below Secondary,m,1,26,5,1,0,0,58,sourcing,0
3,Analytics,Master's & above,m,1,49,4,12,0,0,75,sourcing,0
4,Sales & Marketing,Bachelor's,f,2,48,4,1,1,0,53,other,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (8000, 12)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 2

Target distribution:
is_promoted
0    7320
1     680
Name: count, dtype: int64

Dtypes:
department              object
education               object
gender                  object
no_of_trainings          int32
age                      int64
previous_year_rating     int64
length_of_service        int64
KPIs_met                 int32
awards_won               int32
avg_training_score       int64
recruitment_channel     object
is_promoted              int64
dtype: object

Target 'is_promoted' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(['avg_training_score', 'age', 'length_of_service', 'previous_year_rating', 'no_of_trainings', 'KPIs_met']):
    ax = axes[i // 3, i % 3]
    df[df[TARGET]==0][col].hist(bins=20, ax=ax, alpha=0.6, label='Not Promoted', color='steelblue')
    df[df[TARGET]==1][col].hist(bins=20, ax=ax, alpha=0.6, label='Promoted', color='coral')
    ax.set_title(col); ax.legend()
plt.suptitle("Feature Distributions by Promotion Status", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['department', 'education']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Promotion Rate by {col}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 7, Categorical: 4


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Promotion Distribution")
axes[0].set_xticklabels(['Not Promoted (0)', 'Promoted (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / max(df[TARGET].value_counts().iloc[1], 1):.0f}:1")

Class distribution:
is_promoted
0    7320
1     680
Name: count, dtype: int64

Imbalance ratio: 11:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (6400, 11), Test: (1600, 11)
Train target dist: {0: 5856, 1: 544}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (11): ['department', 'education', 'gender', 'no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'KPIs_met', 'awards_won', 'avg_training_score', 'recruitment_channel']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.9331
Precision: 0.9260
Recall   : 0.9331
F1       : 0.9178
ROC-AUC  : 0.8462


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.902500           0.716610  0.845618  0.905224   0.908319  0.902500    0.018433
LogisticRegression             0.933125           0.636632  0.846085  0.917849   0.925984  0.933125    0.019874
AdaBoostClassifier             0.931875           0.632614  0.842977  0.916314   0.923586  0.931875    0.176190
RidgeClassifier                0.931875           0.632614  0.845789  0.916314   0.923586  0.931875    0.017718
RidgeClassifierCV              0.931875           0.632614  0.845789  0.916314   0.923586  0.931875    0.020693
SGDClassifier                  0.931875           0.632614  0.802334  0.916314   0.923586  0.931875    0.035290
QuadraticDiscriminantAnalysis  0.931875           0.632614  0.842635  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
_flaml_metric = "macro_f1" if y_train.nunique() > 2 else "f1"
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric=_flaml_metric,
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.9256
F1       : 0.9165


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.9300  F1: 0.9143


LightGBM  Acc: 0.9244  F1: 0.9104


XGBoost   Acc: 0.9250  F1: 0.9104


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.9331  0.9178     0.9260  0.9331
FLAML                  0.9256  0.9165     0.9137  0.9256
CatBoost               0.9300  0.9143     0.9198  0.9300
XGBoost                0.9250  0.9104     0.9104  0.9250
LightGBM               0.9244  0.9104     0.9095  0.9244

Top 3: ['Logistic Regression', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  Logistic Regression
              precision    recall  f1-score   support

           0       0.94      0.99      0.96      1464
           1       0.81      0.28      0.42       136

    accuracy                           0.93      1600
   macro avg       0.87      0.64      0.69      1600
weighted avg       0.93      0.93      0.92      1600


  FLAML
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      1464
           1       0.61      0.35      0.45       136

    accuracy                           0.93      1600
   macro avg       0.77      0.67      0.70      1600
weighted avg       0.91      0.93      0.92      1600


  CatBoost
              precision    recall  f1-score   support

           0       0.94      0.99      0.96      1464
           1       0.75      0.26      0.39       136

    accuracy                           0.93      1600
   macro avg       0.84      0.63      0.6

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: Logistic Regression
Error rate: 0.0669 (107 / 1600)

Errors by true class:
      errors  total  error_rate
true                           
0          9   1464    0.006148
1         98    136    0.720588


## 25 · Interpretation and Business Insight

- **KPIs met** is the strongest promotion predictor — performance matters most.
- **Awards won** is rare but highly predictive of promotion.
- **Average training score** above ~70 increases promotion probability.
- **Previous year rating** of 4-5 correlates with promotion.
- **Department** and **education** have moderate effects.
- **Gender** and **age** are weak predictors — good for fairness.

## 26 · Limitations

1. Synthetic data — real promotion decisions involve subjective factors.
2. Only 11 features — missing manager assessments, project outcomes.
3. ~8.5% promotion rate means heavy class imbalance.
4. No temporal data (multi-year career progression).
5. Potential bias in historical promotion patterns.

## 27 · How to Improve This Project

1. Apply class weights for better recall on promoted employees.
2. Add manager ratings and 360-degree feedback features.
3. Engineer tenure × performance interaction features.
4. Conduct fairness audit across gender and age groups.
5. Use SHAP for individual promotion explanations.

## 28 · Production Considerations

- Tool for talent management, not automated promotion decisions.
- Must audit for disparate impact across protected groups.
- Integrate with performance management systems.
- Regular retraining as organizational priorities change.
- Employees should know the criteria — transparency matters.

## 29 · Common Mistakes

1. Using accuracy on 91.5/8.5 split (91.5% baseline!).
2. Not auditing for gender or age bias in predictions.
3. Treating model scores as entitlement to promotion.
4. Ignoring that promotion criteria change over time.
5. Not considering organizational budget constraints.

## 30 · Mini Challenge / Exercises

1. Audit the model for gender bias — does prediction rate differ by gender?
2. What's the impact of removing KPIs_met? How much does F1 drop?
3. Build a rule-based system (KPIs + awards + rating) and compare to ML.
4. Calculate promotion probability for a specific employee profile.

## 31 · Final Summary / Key Takeaways

1. Employee promotion prediction is a practical HR analytics problem.
2. KPI achievement and awards are the strongest promotion signals.
3. Class imbalance (~8.5%) requires careful metric selection.
4. Fairness auditing is essential for HR ML applications.
5. Models should support, not replace, human promotion decisions.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.93,
    "f1": 0.9143,
    "precision": 0.9198,
    "recall": 0.93
  },
  "LightGBM": {
    "accuracy": 0.9244,
    "f1": 0.9104,
    "precision": 0.9095,
    "recall": 0.9244
  },
  "XGBoost": {
    "accuracy": 0.925,
    "f1": 0.9104,
    "precision": 0.9104,
    "recall": 0.925
  },
  "Logistic Regression": {
    "accuracy": 0.9331,
    "f1": 0.9178,
    "precision": 0.926,
    "recall": 0.9331
  },
  "FLAML": {
    "accuracy": 0.9256,
    "f1": 0.9165,
    "precision": 0.9137,
    "recall": 0.9256
  }
}
